# Newton's Second Law

This version keeps each experiment self-contained: the controls and animation appear together.

It uses ordinary Matplotlib objects only: `Rectangle`, `ax.arrow()`, `ax.text()` and a simple frame loop.

## 1. Motion demo

Change $F$ and $m$. The acceleration is

$$a=\frac{F}{m}$$

Press **Run** to watch the block move.


In [1]:
%pip install pandas
%pip install ipywidgets  # needed by the in-browser (Pyodide) kernel
import io
import time
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, FancyArrowPatch
import ipywidgets as widgets
from IPython.display import display, clear_output, Image


def fig_to_image(fig):
    """Render a Matplotlib figure to a FIXED-SIZE PNG.sions and only the block itself appears to move.
    """
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=fig.dpi,
                bbox_inches=None, facecolor="white")
    buf.seek(0)
    return Image(data=buf.getvalue())


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [4]:
# ============================================================
# DEMO 1: vary F and m, then watch the block move
# Everything for this demo is in this one cell.
# ============================================================

force_slider = widgets.IntSlider(
    value=100, min=20, max=300, step=20,
    description="Force F (N):",
    continuous_update=False
)

mass_slider = widgets.IntSlider(
    value=20, min=5, max=60, step=5,
    description="Mass m (kg):",
    continuous_update=False
)

run_button = widgets.Button(
    description="Run",
    button_style="primary",
    icon="play"
)

reset_button = widgets.Button(description="Reset")
value_label = widgets.HTML()
sim_output = widgets.Output(layout=widgets.Layout(width="700px"))

# --- Build the figure once, and keep handles to every artist we
# --- need to move later, instead of rebuilding it every frame. ---
try:
    plt.close(fig)   # in case this cell gets re-run
except NameError:
    pass

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.set_xlim(0, 25)
ax.set_ylim(-1, 6)
ax.axhline(1, color="black", linewidth=2)

block = Rectangle((0, 1), 2.5, 1.6, facecolor="lightblue",
                   edgecolor="black", linewidth=2)
ax.add_patch(block)
mass_text = ax.text(1.25, 1.8, "", ha="center", va="center", fontsize=13)

force_arrow = FancyArrowPatch((2.7, 1.8), (5.7, 1.8),
                               arrowstyle="-|>", mutation_scale=20,
                               linewidth=2, color="black")
ax.add_patch(force_arrow)
force_text = ax.text(4.2, 2.25, "", ha="center", fontsize=12)

accel_arrow = FancyArrowPatch((0.2, 4.1), (4.2, 4.1),
                               arrowstyle="-|>", mutation_scale=22,
                               linewidth=2.5, color="firebrick")
ax.add_patch(accel_arrow)
accel_text = ax.text(2.2, 4.65, "", ha="center", fontsize=13, fontweight="bold")

ax.set_title("Block motion")
ax.set_xlabel("Position")
ax.set_yticks([])
for side in ("left", "right", "top"):
    ax.spines[side].set_visible(False)
fig.tight_layout()
plt.close(fig)


def update_frame(x=0.0):
    """Move every artist to reflect the current x, F, m, a and redraw."""
    F = force_slider.value
    m = mass_slider.value
    a = F / m

    value_label.value = (
        f"<b>F = {F} N</b><br>"
        f"<b>m = {m} kg</b><br>"
        f"<b>a = {a:.2f} m/s\u00b2</b>"
    )

    block.set_x(x)
    mass_text.set_position((x + 1.25, 1.8))
    mass_text.set_text(f"{m} kg")

    force_arrow.set_positions((x + 2.7, 1.8), (x + 5.7, 1.8))
    force_text.set_position((x + 4.2, 2.25))
    force_text.set_text(f"F = {F} N")

    accel_arrow.set_positions((x + 0.2, 4.1), (x + 4.2, 4.1))
    accel_text.set_position((x + 2.2, 4.65))
    accel_text.set_text(f"a = {a:.2f} m/s\u00b2")

    with sim_output:
        clear_output(wait=True)
        display(fig_to_image(fig))


state = {"running": False}


def run_demo_animation():
    # Plain synchronous loop (no asyncio). Button callbacks in
    # ipywidgets are not guaranteed to run inside a live asyncio
    # event loop in every Jupyter front end (this is what broke
    # the animation), so a blocking time.sleep() loop is the more
    # portable choice -- it works the same everywhere.
    if state["running"]:
        return
    state["running"] = True

    F = force_slider.value
    m = mass_slider.value
    a = F / m

    run_button.disabled = True
    force_slider.disabled = True
    mass_slider.disabled = True

    try:
        times = np.linspace(0, 2.0, 35)
        for t in times:
            x = min(21.5, 0.5 * a * t**2)
            update_frame(x)
            time.sleep(0.05)
    finally:
        run_button.disabled = False
        force_slider.disabled = False
        mass_slider.disabled = False
        state["running"] = False


def run_clicked(button):
    run_demo_animation()


def reset_clicked(button):
    update_frame(0.0)


def controls_changed(change=None):
    if not state["running"]:
        update_frame(0.0)


run_button.on_click(run_clicked)
reset_button.on_click(reset_clicked)
force_slider.observe(controls_changed, names="value")
mass_slider.observe(controls_changed, names="value")

controls = widgets.VBox([
    widgets.HTML("<h3>Controls</h3>"),
    force_slider,
    mass_slider,
    widgets.HBox([run_button, reset_button]),
    value_label
], layout=widgets.Layout(width="320px"))

# Controls and simulation are deliberately displayed together.
display(widgets.HBox([
    controls,
    sim_output
], layout=widgets.Layout(align_items="flex-start", gap="18px")))

# Show the box immediately, before Run is pressed.
update_frame(0.0)


## 2. Build an $F$ versus $a$ graph

For this experiment the mass is fixed.

Change only $F$, press **Run + record**, and collect the pair $(a,F)$.

Because

$$F=ma$$

a plot of $F$ against $a$ should be a straight line. Its slope is the mass.


In [3]:
# ============================================================
# DEMO 2: fixed mass, vary F, record (a, F) pairs
# Everything for this experiment is in this one cell.
# Same fix as Demo 1: build the block figure once and move its
# artists each frame, plus a "running" guard on Run + record.
# ============================================================

MASS = 20  # kg -- students can change this one number

force_exp = widgets.IntSlider(
    value=200, min=20, max=300, step=20,
    description="Force F (N):",
    continuous_update=False
)

run_record = widgets.Button(
    description="Run + record",
    button_style="primary",
    icon="play"
)

reset_exp = widgets.Button(description="Reset block")
clear_data = widgets.Button(description="Clear data")
exp_value_label = widgets.HTML()

exp_output = widgets.Output(layout=widgets.Layout(width="700px"))
graph_output = widgets.Output(layout=widgets.Layout(width="760px"))
show_fit = widgets.Checkbox(
    value=False,
    description="Show fit line"
)

F_values = []
a_values = []

# --- Build the block figure once (same reasoning as Demo 1) ---
try:
    plt.close(exp_fig)
except NameError:
    pass

exp_fig, exp_ax = plt.subplots(figsize=(8, 3.2))
exp_ax.set_xlim(0, 25)
exp_ax.set_ylim(0, 6)
exp_ax.axhline(1, color="black", linewidth=2)

exp_block = Rectangle((0, 1), 2.5, 1.6, facecolor="lightblue",
                       edgecolor="black", linewidth=2)
exp_ax.add_patch(exp_block)
exp_mass_text = exp_ax.text(1.25, 1.8, f"{MASS} kg", ha="center", va="center", fontsize=13)

exp_force_arrow = FancyArrowPatch((2.7, 1.8), (5.7, 1.8),
                                   arrowstyle="-|>", mutation_scale=20,
                                   linewidth=2, color="black")
exp_ax.add_patch(exp_force_arrow)
exp_force_text = exp_ax.text(4.2, 2.25, "", ha="center", fontsize=12)

exp_accel_arrow = FancyArrowPatch((0.2, 4.1), (4.2, 4.1),
                                   arrowstyle="-|>", mutation_scale=22,
                                   linewidth=2.5, color="firebrick")
exp_ax.add_patch(exp_accel_arrow)
exp_accel_text = exp_ax.text(2.2, 4.65, "", ha="center", fontsize=13, fontweight="bold")

exp_ax.set_title("Fixed-mass experiment")
exp_ax.set_xlabel("Position")
exp_ax.set_yticks([])
for side in ("left", "right", "top"):
    exp_ax.spines[side].set_visible(False)
exp_fig.tight_layout()
plt.close(exp_fig)

def update_experiment_frame(x=0.0):
    F = force_exp.value
    a = F / MASS

    exp_value_label.value = (
        f"<b>Fixed mass = {MASS} kg</b><br>"
        f"<b>F = {F} N</b><br>"
        f"<b>a = {a:.2f} m/s\u00b2</b>"
    )

    exp_block.set_x(x)
    exp_mass_text.set_position((x + 1.25, 1.8))

    exp_force_arrow.set_positions((x + 2.7, 1.8), (x + 5.7, 1.8))
    exp_force_text.set_position((x + 4.4, 2.25))
    exp_force_text.set_text(f"F = {F} N")

    exp_accel_arrow.set_positions((x + 0.2, 4.1), (x + 4.2, 4.1))
    exp_accel_text.set_position((x + 2.2, 4.65))
    exp_accel_text.set_text(f"a = {a:.2f} m/s\u00b2")

    with exp_output:
        clear_output(wait=True)
        display(fig_to_image(exp_fig))


def draw_F_a_graph():
    with graph_output:
        clear_output(wait=True)

        fig2, ax2 = plt.subplots(figsize=(7, 4.5))
        ax2.scatter(a_values, F_values, s=70)
        
        if show_fit.value and len(a_values) >= 2:
            slope, intercept = np.polyfit(a_values, F_values, 1)
            xmax = max(a_values) * 1.12
            xx = np.linspace(0, xmax, 100)
            ax2.plot(xx, slope * xx + intercept)
            ax2.text(
                0.05, 0.92,
                f"slope = {slope:.1f} kg",
                transform=ax2.transAxes,
                fontsize=12
            )

        ax2.set_xlim(left=0)
        ax2.set_ylim(bottom=0)
        ax2.set_xlabel("Acceleration, a (m/s\u00b2)")
        ax2.set_ylabel("Force, F (N)")
        ax2.set_title("F versus a")
        ax2.grid(alpha=0.3)
        plt.show()
        plt.close(fig2)


exp_state = {"running": False}


def run_and_record():
    # Synchronous loop -- see the note in the Demo 1 cell above
    # about why asyncio.create_task() was replaced.
    if exp_state["running"]:
        return
    exp_state["running"] = True

    F = force_exp.value
    a = F / MASS

    run_record.disabled = True
    force_exp.disabled = True

    try:
        times = np.linspace(0, 2.0, 35)
        for t in times:
            x = min(21.5, 0.5 * a * t**2)
            update_experiment_frame(x)
            time.sleep(0.05)

        # Record exactly one pair at the end of the run.
        a_values.append(a)
        F_values.append(F)
        draw_F_a_graph()
    finally:
        run_record.disabled = False
        force_exp.disabled = False
        exp_state["running"] = False


def run_record_clicked(button):
    run_and_record()


def reset_exp_clicked(button):
    update_experiment_frame(0.0)


def clear_data_clicked(button):
    F_values.clear()
    a_values.clear()
    draw_F_a_graph()


def force_exp_changed(change=None):
    if not exp_state["running"]:
        update_experiment_frame(0.0)


run_record.on_click(run_record_clicked)
reset_exp.on_click(reset_exp_clicked)
clear_data.on_click(clear_data_clicked)
show_fit.observe(lambda change: draw_F_a_graph(), names="value")
force_exp.observe(force_exp_changed, names="value")

exp_controls = widgets.VBox([
    widgets.HTML("<h3>Controls</h3>"),
    force_exp,
    widgets.HBox([run_record, reset_exp]),
    clear_data,
    show_fit,
    exp_value_label
], layout=widgets.Layout(width="320px"))

# Show both outputs immediately.
update_experiment_frame(0.0)
# draw_F_a_graph()

display(widgets.VBox([
    widgets.HBox([exp_controls, exp_output],
                 layout=widgets.Layout(align_items="flex-start", gap="18px")),
    graph_output
]))
